# 004_postprocess_across_temporal_condition_specific.ipynb

Across-condition temporal postprocessing branch using condition-specific decoding rankings. For across models, selected/top archetypes are taken from the union of condition-specific top archetypes instead of a single ranking averaged across conditions.


In [ ]:
# ============================================================
# FIGURE SAVING SETTINGS -- EXPLICIT, NO RECURSION
# ============================================================

from pathlib import Path

SAVE_FIGS = True
FIG_ROOT = "/Users/lowen/Desktop/papers/archetypes/figures"
FIG_NOTEBOOK_DIR = "004_postprocess_across_temporal"
FIG_DIR = Path(FIG_ROOT) / FIG_NOTEBOOK_DIR
FIG_FORMAT = "pdf"
DPI = 300

FIG_DIR.mkdir(parents=True, exist_ok=True)

_fig_counter = 0

def _sanitize_fig_name(name):
    name = str(name).replace(" ", "_").replace("|", "_").replace("/", "-").replace("\\", "-")
    name = "".join(ch for ch in name if ch.isalnum() or ch in ["_", "-", "."])
    return name[:160] if name else "figure"

def _figure_has_content(fig=None):
    if fig is None:
        fig = plt.gcf()
    if len(fig.axes) == 0:
        return False
    for ax in fig.axes:
        if ax.lines or ax.collections or ax.images or ax.patches or ax.texts or ax.get_title():
            return True
    return True

def save_current_fig(name=None):
    """
    Save current matplotlib figure. This does NOT patch plt.show, so it cannot recurse.
    """
    global _fig_counter

    if not SAVE_FIGS:
        return None

    fig = plt.gcf()
    if not _figure_has_content(fig):
        return None

    _fig_counter += 1

    if name is None:
        try:
            title = plt.gca().get_title()
        except Exception:
            title = ""
        label = _sanitize_fig_name(title if title else "figure")
    else:
        label = _sanitize_fig_name(name)

    out = FIG_DIR / f"{_fig_counter:03d}_{label}.{FIG_FORMAT}"
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    print("Saved:", out)
    return out

def savefig(name=None, force=True):
    return save_current_fig(name=name)

def show_save_close(name=None):
    save_current_fig(name)
    plt.show()
    plt.close()

print("Figure directory:", FIG_DIR)
print("Explicit save mode: plt.show is not patched.")


In [ ]:
from pathlib import Path

%matplotlib inline
import os, warnings
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from scipy.io import loadmat
from sklearn.cluster import SpectralClustering
from matplotlib import cm, colors as mcolors

os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    import nibabel as nib
    from nilearn.input_data import NiftiMasker
    from nilearn import plotting as niplot
    NILEARN_AVAILABLE = True
except Exception:
    NILEARN_AVAILABLE = False
    print("nilearn / nibabel not available; brain plotting disabled.")

print("Ready.")

In [ ]:

FIT_LOAD_DIR = "msaa_flexible_outputs_npz"
DECODE_LOAD_DIR = "msaa_condrank_decoding_outputs_temporal_across"

K_VALUES = [5, 10, 25, 50, 75, 100]
TOP_N_REPORT = 5

# Clustering score tuning
PURITY_WEIGHT = 1.5   # raise this to favor purity more
BALANCE_WEIGHT = 1  # lower this if you want purity to dominate balance
PENALTY_MODE = "weak_sqrt"  # "sqrt", "weak_sqrt", "linear", or "none" # "sqrt", "linear", or "none"

SHOW_SUBJECT_IDS = True

SCHAEFER_TXT = "data/pieman/raw/Schaefer2018_1000Parcels_7Networks_order.txt"
SCHAEFER_NII = "data/pieman/raw/Schaefer2018_1000Parcels_7Networks_order_FSLMNI152_2mm.nii.gz"
POSTERIOR_MAT = "data/pieman/raw/pieman_posterior_K700.mat"

COND_NAME_COLORS = {"intact": "purple", "word": "green", "rest": "black"}

OUTPUT_DIR = "004_postprocess_across_temporal_condition_specific_outputs"


# ============================================================
# SAVED DECODING SUMMARY SETTINGS
# ============================================================

# Do NOT recompute decoding in this postprocessing notebook.
# Instead, read saved per-archetype decoding values from notebook 002.
USE_SAVED_DECODING = True
DECODING_OUTPUT_DIR = "msaa_condrank_decoding_outputs_temporal_across"
PER_ARCH_DECODING_CSV = os.path.join(DECODING_OUTPUT_DIR, "per_archetype_mean_accuracy.csv")

# ============================================================
# BRAIN PLOT SETTINGS
# ============================================================

RUN_BRAIN_PLOTS = True
BRAIN_PLOT_MODE = "markers"   # "markers", "scatter", or "none"
MAX_BRAIN_NODES = 150
BRAIN_NODE_SIZE_MIN = 12
BRAIN_NODE_SIZE_MAX = 70
CLOSE_BRAIN_FIGURES = True

In [ ]:

def to_float_array(x):
    return np.array(x, dtype=float)

def load_msaa_npz(path):
    data = np.load(path, allow_pickle=True)
    results_subj = data["results_subj"].tolist()
    if isinstance(results_subj, np.ndarray):
        results_subj = results_subj.tolist()
    return {
        "K": int(data["K"]),
        "results_subj": results_subj,
        "condition_labels_str": data["condition_labels_str"].tolist(),
    }

rankings = np.load(os.path.join(DECODE_LOAD_DIR, "rankings.npy"), allow_pickle=True).item()
all_fits = {
    K: load_msaa_npz(os.path.join(FIT_LOAD_DIR, f"temporalAA_across_acrossCond_K{K}.npz"))
    for K in K_VALUES
}
print("Loaded fits and rankings.")

In [ ]:

posterior = loadmat(POSTERIOR_MAT)
centers = to_float_array(posterior['posterior']['centers'][0][0][0][0][0])
widths = to_float_array(list(posterior['posterior']['widths'][0][0][0][0][0][:, 0].T)).ravel()

lookup_table = {
    'Vis':'Visual',
    'SomMot':'Somatomotor',
    'DorsAttn':'Dorsal attention',
    'SalVentAttn':'Ventral attention',
    'Limbic':'Limbic',
    'Cont':'Frontoparietal',
    'Default':'Default mode'
}
network_colors = {
    'Visual':'#D7DF23',
    'Somatomotor':'#39B54A',
    'Dorsal attention':'#00A79D',
    'Ventral attention':'#27AAE1',
    'Limbic':'#1C75BC',
    'Frontoparietal':'#92278F',
    'Default mode':'#EE2A7B'
}
network_codes = {k: i + 1 for i, k in enumerate(lookup_table.values())}

def nii2cmu(nifti_file, mask_file=None):
    def fullfact(dims):
        vals = np.asmatrix(range(1, dims[0] + 1)).T
        if len(dims) == 1:
            return vals
        aftervals = np.asmatrix(fullfact(dims[1:]))
        inds = np.asmatrix(np.zeros((np.prod(dims), len(dims))))
        row = 0
        for i in range(aftervals.shape[0]):
            inds[row:(row + len(vals)), 0] = vals
            inds[row:(row + len(vals)), 1:] = np.tile(aftervals[i, :], (len(vals), 1))
            row += len(vals)
        return inds

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        img = nib.load(nifti_file) if type(nifti_file) == str else nifti_file
        mask = NiftiMasker(mask_strategy='background')
        mask.fit(nifti_file if mask_file is None else mask_file)

    Saff = img.get_sform()
    Y = np.float32(mask.transform(nifti_file)).copy()
    vmask = np.nonzero(np.array(np.reshape(mask.mask_img_.dataobj, (1, np.prod(mask.mask_img_.shape)), order='C')))[1]
    vox_coords = fullfact(img.shape[0:3])[vmask, ::-1] - 1
    R = np.array(np.dot(vox_coords, Saff[0:3, 0:3])) + Saff[:3, 3]
    return {'Y': Y, 'R': R}

def rbf(R, center, width):
    return np.exp(-np.sum((R - center) ** 2, axis=1) / width)

def node_labels(centers, widths, networks_cmu):
    labels = []
    for c, w in zip(centers, widths):
        r = rbf(networks_cmu['R'], c, w)
        label_weights = [sum(r[networks_cmu['Y'].ravel() == i]) for i in range(1, len(network_codes)+1)]
        labels.append(np.argmax(label_weights)+1)
    return pd.DataFrame({
        'code': labels,
        'Network': [list(lookup_table.values())[i-1] for i in labels]
    })

if NILEARN_AVAILABLE:
    key = pd.read_csv(SCHAEFER_TXT, sep='\t', header=None, names=['id','name','x','y','z','t']).drop('t', axis=1)
    key['network'] = key['name'].apply(lambda x: lookup_table[x.split('_')[2]])
    key['code'] = key['network'].apply(lambda x: network_codes[x])
    key.set_index('id', inplace=True)
    key.loc[0, 'code'] = 0
    networks_cmu = nii2cmu(SCHAEFER_NII)
    networks_cmu['Y'] = np.atleast_2d(np.array([key.loc[i, 'code'] for i in networks_cmu['Y']]).astype(float))
    node_code_df = node_labels(centers, widths, networks_cmu)
else:
    node_code_df = None

In [ ]:

def purity_score(y_cluster, y_class):
    total = 0
    for c in np.unique(y_cluster):
        mask = (y_cluster == c)
        _, counts = np.unique(np.asarray(y_class)[mask], return_counts=True)
        total += counts.max()
    return total / len(y_cluster)

def equal_size_balance(labels):
    _, counts = np.unique(labels, return_counts=True)
    N = counts.sum()
    k = len(counts)
    ideal = N / k
    return float(1.0 - np.sum(np.abs(counts - ideal)) / (2 * N))

def cluster_count_penalty(n_clusters, mode="sqrt"):
    if mode == "linear":
        return 1.0 / n_clusters
    elif mode == "sqrt":
        return 1.0 / np.sqrt(n_clusters)
    elif mode == "weak_sqrt":
        return 1.0 / (n_clusters ** 0.25)
    elif mode == "none":
        return 1.0
    else:
        raise ValueError("mode must be 'linear', 'sqrt', 'weak_sqrt', or 'none'")


def combined_cluster_score(purity, balance, n_clusters, purity_weight=1.5, balance_weight=1.0, penalty_mode="sqrt"):
    penalty = cluster_count_penalty(n_clusters, mode=penalty_mode)
    return (purity ** purity_weight) * (balance ** balance_weight) * penalty

def reorder_by_labels_and_within_similarity(sim, labels):
    unique_labels = np.unique(labels)
    order = []
    for lab in unique_labels:
        idx = np.where(labels == lab)[0]
        sub_sim = sim[np.ix_(idx, idx)]
        sub_order = idx[np.argsort(-sub_sim.mean(axis=1))]
        order.extend(sub_order.tolist())
    order = np.array(order)
    labels_sorted = labels[order]
    boundaries = np.where(np.asarray(labels_sorted[1:]) != np.asarray(labels_sorted[:-1]))[0] + 1
    return order, boundaries

def cluster_temporal(results_subj, cond_codes, k, purity_weight=1.5, balance_weight=1.0, penalty_mode="sqrt"):
    Xk = np.stack([to_float_array(sub["sXC"])[:, k] for sub in results_subj], axis=0)
    Xk = (Xk - Xk.mean(axis=1, keepdims=True)) / (Xk.std(axis=1, keepdims=True) + 1e-8)
    Xk = np.nan_to_num(Xk, nan=0.0, posinf=0.0, neginf=0.0)

    sim = np.corrcoef(Xk)
    sim = np.nan_to_num(sim, nan=0.0, posinf=0.0, neginf=0.0)
    sim = np.clip(sim, -1.0, 1.0)
    np.fill_diagonal(sim, 1.0)

    best = None
    scan_rows = []
    for n_clusters in range(2, 9):
        aff = (sim + 1.0) / 2.0
        np.fill_diagonal(aff, 1.0)
        labels = SpectralClustering(
            n_clusters=n_clusters,
            affinity="precomputed",
            assign_labels="kmeans",
            random_state=0
        ).fit_predict(aff) + 1

        purity = purity_score(labels, cond_codes)
        balance = equal_size_balance(labels)
        score = combined_cluster_score(
            purity, balance, n_clusters,
            purity_weight=purity_weight,
            balance_weight=balance_weight,
            penalty_mode=penalty_mode
        )
        scan_rows.append({
            "n_clusters": n_clusters,
            "purity": purity,
            "balance": balance,
            "score": score,
            "cluster_sizes": np.unique(labels, return_counts=True)[1].tolist(),
        })
        if best is None or score > best["score"]:
            best = {
                "score": score,
                "n_clusters": n_clusters,
                "labels": labels,
                "sim": sim,
                "purity": purity,
                "balance": balance,
            }

    order, boundaries = reorder_by_labels_and_within_similarity(best["sim"], best["labels"])
    return {
        "scan_df": pd.DataFrame(scan_rows).sort_values("score", ascending=False).reset_index(drop=True),
        "sim_sorted": best["sim"][np.ix_(order, order)],
        "boundaries": boundaries,
        "order": order,
        "labels": best["labels"],
        "ordered_labels": np.asarray(best["labels"])[order],
        "best_n_clusters": best["n_clusters"],
        "purity": best["purity"],
        "balance": best["balance"],
    }

def plot_cluster_scan(cluster_res, K, k):
    df = cluster_res["scan_df"].sort_values("n_clusters")
    plt.figure(figsize=(8, 5))
    plt.plot(df["n_clusters"], df["purity"], marker="o", label="Purity")
    plt.plot(df["n_clusters"], df["balance"], marker="o", label="Balance")
    plt.plot(df["n_clusters"], df["score"], marker="o", label="Combined score")
    plt.xlabel("Number of clusters")
    plt.ylabel("Score")
    plt.title(f"K={K} | archetype {k} | cluster-number scan")
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_cluster_similarity(cluster_res, cond_labels, K, k, show_subject_ids=True):
    order = cluster_res["order"]
    cond_sorted = np.asarray(cond_labels)[order]
    xticklabels = (order + 1) if show_subject_ids else False
    yticklabels = (order + 1) if show_subject_ids else False

    plt.figure(figsize=(7, 6))
    ax = sns.heatmap(
        cluster_res["sim_sorted"],
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
        square=True,
        xticklabels=xticklabels,
        yticklabels=yticklabels,
        cbar_kws={"label": "subject correlation of spatial motifs"}
    )
    if show_subject_ids:
        tick_colors = [{"intact":"purple","word":"green","rest":"black"}.get(c, "gray") for c in cond_sorted]
        for tick_label, color in zip(ax.get_xticklabels(), tick_colors):
            tick_label.set_color(color)
            tick_label.set_rotation(90)
            tick_label.set_fontsize(8)
        for tick_label, color in zip(ax.get_yticklabels(), tick_colors):
            tick_label.set_color(color)
            tick_label.set_fontsize(8)

    for b in cluster_res["boundaries"]:
        ax.axhline(b, color="black", linewidth=2)
        ax.axvline(b, color="black", linewidth=2)

    plt.title(
        f"K={K} | archetype {k} | clustering on spatial motifs\n"
        f"best_n={cluster_res['best_n_clusters']}, purity={cluster_res['purity']:.2f}, balance={cluster_res['balance']:.2f}"
    )
    plt.tight_layout()
    plt.show()

def plot_coeff_timecourses(results_subj, cond_labels, k, K):
    rows = []
    for i, sub in enumerate(results_subj):
        coeff = to_float_array(sub["S"])[k, :]
        rows.append(pd.DataFrame({"time": np.arange(len(coeff)), "value": coeff, "condition": cond_labels[i]}))
    df = pd.concat(rows, ignore_index=True)

    plt.figure(figsize=(9, 4))
    for cond_name, color in {"intact":"purple","word":"green","rest":"black"}.items():
        cur = df[df["condition"] == cond_name]
        if len(cur) == 0:
            continue
        summ = cur.groupby("time")["value"].agg(["mean", "std", "count"]).reset_index()
        summ["sem"] = summ["std"] / np.sqrt(summ["count"].clip(lower=1))
        plt.plot(summ["time"], summ["mean"], color=color, label=cond_name)
        plt.fill_between(summ["time"], summ["mean"] - summ["sem"], summ["mean"] + summ["sem"], color=color, alpha=0.2)
    plt.title(f"K={K} | archetype {k} | coefficient timecourses")
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_subject_heatmap(results_subj, cond_labels, k, K, order=None, cluster_boundaries=None, cluster_labels=None, show_subject_ids=True):
    Xk = np.stack([to_float_array(sub["S"])[k, :] for sub in results_subj], axis=0)
    if order is None:
        order = np.arange(Xk.shape[0])
    Xk = Xk[order]
    cond_sorted = np.asarray(cond_labels)[order]
    yticklabels = (order + 1) if show_subject_ids else False

    plt.figure(figsize=(10, 5))
    ax = sns.heatmap(Xk, cmap="coolwarm", center=0, yticklabels=yticklabels)

    if show_subject_ids:
        for tick_label, cond in zip(ax.get_yticklabels(), cond_sorted):
            tick_label.set_color({"intact":"purple","word":"green","rest":"black"}.get(cond, "gray"))
            tick_label.set_fontsize(8)

    if cluster_boundaries is not None:
        for b in cluster_boundaries:
            ax.axhline(b, color="black", linewidth=2.0)

    if cluster_labels is not None and cluster_boundaries is not None:
        starts = np.r_[0, cluster_boundaries]
        ends = np.r_[cluster_boundaries, len(order)]
        mids = (starts + ends) / 2.0
        ordered_labels = np.asarray(cluster_labels)[order]
        block_labels = []
        for s, e in zip(starts, ends):
            block = ordered_labels[s:e]
            block_labels.append(int(pd.Series(block).mode().iloc[0]))
        for mid, lab in zip(mids, block_labels):
            ax.text(-0.5, mid, f"C{lab}", va="center", ha="right", fontsize=9, fontweight="bold", color="black", clip_on=False)

    plt.title(f"K={K} | archetype {k} | subject × time heatmap (cluster-ordered)")
    plt.tight_layout()
    plt.show()

def plot_spatial_map(results_subj, k, K):
    """
    Original temporal-AA spatial motif brain plot.

    Temporal AA:
        sXC is V x K, so sXC[:, k] is the spatial motif.
    """
    if not NILEARN_AVAILABLE:
        print("Nilearn not available; skipping brain plot.")
        return

    vals = np.stack(
        [to_float_array(sub["sXC"])[:, k] for sub in results_subj],
        axis=0
    ).mean(axis=0)

    vals = np.nan_to_num(vals)

    vmax = np.max(np.abs(vals))
    if vmax == 0 or not np.isfinite(vmax):
        vmax = 1.0

    norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)
    cmap = cm.get_cmap("coolwarm")

    node_colors = [cmap(norm(v)) for v in vals]
    node_sizes = 8 + 18 * (np.abs(vals) / (vmax + 1e-8))

    disp = niplot.plot_connectome(
        np.eye(centers.shape[0]),
        centers,
        node_color=node_colors,
        node_size=node_sizes,
        display_mode="lyrz",
        title=f"K={K} | archetype {k} | signed spatial map"
    )

    # Save explicitly without monkey-patching plt.show
    if "save_current_fig" in globals():
        save_current_fig(f"original_brainmap_K{K}_arch{k}")

    plt.show()
    plt.close()

    try:
        disp.close()
    except Exception:
        pass


def plot_network_bar(results_subj, k, K):
    if node_code_df is None:
        return
    vals = np.stack([to_float_array(sub["sXC"])[:, k] for sub in results_subj], axis=0).mean(axis=0)
    vals = np.nan_to_num(vals)
    network_df = node_code_df.copy()
    network_df["value"] = vals
    means = network_df.groupby("Network")["value"].mean().reindex(list(network_colors.keys()))
    plt.figure(figsize=(7.5, 4))
    plt.bar(means.index, means.values, color=[network_colors[n] for n in means.index])
    plt.axhline(0, color="black")
    plt.xticks(rotation=35, ha="right")
    plt.title(f"K={K} | archetype {k} | network mean values")
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# LOAD SAVED PER-ARCHETYPE DECODING
# ============================================================

def load_per_archetype_decoding(path=PER_ARCH_DECODING_CSV):
    if not os.path.exists(path):
        print("Saved per-archetype decoding CSV not found:", path)
        return pd.DataFrame()

    df = pd.read_csv(path)

    rename = {}
    if "mean_accuracy" in df.columns and "mean" not in df.columns:
        rename["mean_accuracy"] = "mean"
    if "sem_accuracy" in df.columns and "sem" not in df.columns:
        rename["sem_accuracy"] = "sem"
    if "err" in df.columns and "sem" not in df.columns:
        rename["err"] = "sem"
    df = df.rename(columns=rename)

    if "analysis_type" in df.columns:
        df = df[df["analysis_type"].astype(str) == "temporal"].copy()
    if "fit_scope" in df.columns:
        df = df[df["fit_scope"].astype(str) == "across"].copy()

    if "K" in df.columns:
        df["K"] = df["K"].astype(int)
    if "archetype" in df.columns:
        df["archetype"] = df["archetype"].astype(int)
    if "condition" in df.columns:
        df["condition"] = df["condition"].astype(str)

    print("Loaded saved per-archetype decoding:", path)
    print("Rows:", len(df))
    if len(df):
        display(df.head())

    return df

per_arch_decoding_df = load_per_archetype_decoding()

In [ ]:

def plot_decoding_bar_by_condition(results_subj, cond_labels, k, K):
    """
    Plot condition-specific single-archetype decoding from saved notebook-002 outputs.

    This replaces the old behavior that recomputed cross-validated decoding inside
    the plotting loop. Recomputing here is slow and unnecessary.
    """
    if USE_SAVED_DECODING:
        if "per_arch_decoding_df" not in globals() or len(per_arch_decoding_df) == 0:
            print(f"No saved decoding available for K={K}, archetype={k}; skipping decoding bar.")
            return

        dec_df = per_arch_decoding_df[
            (per_arch_decoding_df["K"].astype(int) == int(K)) &
            (per_arch_decoding_df["archetype"].astype(int) == int(k))
        ].copy()

        if len(dec_df) == 0:
            print(f"No saved decoding rows for K={K}, archetype={k}; skipping decoding bar.")
            return

        if "mean_accuracy" in dec_df.columns and "mean" not in dec_df.columns:
            dec_df = dec_df.rename(columns={"mean_accuracy": "mean"})
        if "sem_accuracy" in dec_df.columns and "sem" not in dec_df.columns:
            dec_df = dec_df.rename(columns={"sem_accuracy": "sem"})
        if "err" in dec_df.columns and "sem" not in dec_df.columns:
            dec_df = dec_df.rename(columns={"err": "sem"})

        order = [c for c in ["intact", "word", "rest"] if c in set(dec_df["condition"].astype(str))]
        dec_df["condition"] = pd.Categorical(dec_df["condition"].astype(str), categories=order, ordered=True)
        dec_df = dec_df.sort_values("condition")

        yerr = dec_df["sem"] if "sem" in dec_df.columns else None

        plt.figure(figsize=(5, 4))
        plt.bar(
            dec_df["condition"].astype(str),
            dec_df["mean"],
            yerr=yerr,
            color=[COND_NAME_COLORS.get(str(c), "gray") for c in dec_df["condition"].astype(str)],
            capsize=4
        )
        plt.ylabel("Decoding accuracy")
        plt.title(f"K={K} | archetype {k} | saved single-archetype decoding")
        plt.tight_layout()
        show_save_close(f"saved_decoding_bar_K{K}_arch{k}")
        return

    raise RuntimeError("USE_SAVED_DECODING=False would recompute decoding. Use notebook 002 outputs instead.")




## Note on the coefficient heatmap

The coefficient heatmap now uses the **same clustering solution** as the similarity matrix:

- same subject order
- same cluster boundaries

So the horizontal separators now indicate **cluster membership**.
Condition identity is shown by the **color of the subject labels**.


In [ ]:

for K in K_VALUES:
    cur = all_fits[K]
    results_subj = cur["results_subj"]
    cond_labels = np.array(cur["condition_labels_str"])
    cond_codes = pd.Categorical(cond_labels, categories=sorted(np.unique(cond_labels))).codes

    # Condition-specific ranking branch:
    # across-condition top archetypes are the union of each condition's top archetypes.
    top_arch_set = []
    for cond_name in ["intact", "word", "rest"]:
        key = (cond_name, K)
        if key in rankings:
            top_arch_set.extend(list(rankings[key][:TOP_N_REPORT]))
    if len(top_arch_set) == 0:
        raise KeyError(f"No condition-specific rankings found for K={K}. Available keys: {list(rankings.keys())[:10]}")
    top_arches = sorted(set(top_arch_set))[:TOP_N_REPORT]
    print(f"\nK={K} | top archetypes:", top_arches)

    for k in top_arches:
        cluster_res = cluster_temporal(
            results_subj, cond_codes, k,
            purity_weight=PURITY_WEIGHT,
            balance_weight=BALANCE_WEIGHT,
            penalty_mode=PENALTY_MODE
        )

        plot_cluster_scan(cluster_res, K, k)
        plot_cluster_similarity(cluster_res, cond_labels, K, k, show_subject_ids=SHOW_SUBJECT_IDS)
        plot_decoding_bar_by_condition(results_subj, cond_labels, k, K)
        plot_coeff_timecourses(results_subj, cond_labels, k, K)
        plot_subject_heatmap(
            results_subj, cond_labels, k, K,
            order=cluster_res["order"],
            cluster_boundaries=cluster_res["boundaries"],
            cluster_labels=cluster_res["labels"],
            show_subject_ids=SHOW_SUBJECT_IDS
        )
        plot_spatial_map(results_subj, k, K)
        plot_network_bar(results_subj, k, K)